# Демо 1. Класичне донавчання: зміна голови в моделі BERT

У цьому ноутбуці донавчаємо модель **BERT** для задачі **визначення парафраз**: модель отримує два речення й визначає, чи передають вони той самий зміст.

План:
1. Завантажити датасет MRPC і токенізувати пари речень.
2. Завантажити BERT із **новою головою**-класифікатором.
3. **Повне донавчання** з `Trainer`: тренуються всі ваги (кістяк і голова).
4. Перевірити модель на власних прикладах.

Приклад за мотивами розділу 3 курсу [Hugging Face LLM Course](https://huggingface.co/learn/llm-course/chapter3/1).
Для запуску в Colab оберіть середовище з GPU: *Runtime → Change runtime type → T4 GPU*.

In [1]:
import numpy as np
import torch
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)

In [2]:
# Фіксуємо seed, щоб результати повторювалися між запусками
set_seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Пристрій: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name()}")
else:
    print("Включіть GPU, бо набридне чекати ;)")

Пристрій: cuda
GPU: Tesla T4


## 1. Дані: MRPC

**MRPC** (Microsoft Research Paraphrase Corpus) — пари речень з новинних статей.
Мітка `1` означає, що речення є парафразами, мітка `0` — що ні.

In [3]:
raw_datasets = load_dataset("nyu-mll/glue", "mrpc")
raw_datasets

README.md:   0%|          | 0.00/35.3k [00:00<?, ?B/s]

mrpc/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  649kB            

mrpc/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

mrpc/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 75.7kB            

mrpc/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

mrpc/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  308kB            

mrpc/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/3668 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/408 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1725 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1725
    })
})

In [4]:
# Подивимося на один приклад
print("Приклад нееквівалентних речень:")
example = raw_datasets["train"][3]
print("Речення 1:", example["sentence1"])
print("Речення 2:", example["sentence2"])
print("Мітка:   ", example["label"], "→", raw_datasets["train"].features["label"].names[example["label"]])

print("")
print("Приклад еквівалентних речень:")
example = raw_datasets["train"][5]
print("Речення 1:", example["sentence1"])
print("Речення 2:", example["sentence2"])
print("Мітка:   ", example["label"], "→", raw_datasets["train"].features["label"].names[example["label"]])

Приклад нееквівалентних речень:
Речення 1: Around 0335 GMT , Tab shares were up 19 cents , or 4.4 % , at A $ 4.56 , having earlier set a record high of A $ 4.57 .
Речення 2: Tab shares jumped 20 cents , or 4.6 % , to set a record closing high at A $ 4.57 .
Мітка:    0 → not_equivalent

Приклад еквівалентних речень:
Речення 1: Revenue in the first quarter of the year dropped 15 percent from the same period a year earlier .
Речення 2: With the scandal hanging over Stewart 's company , revenue the first quarter of the year dropped 15 percent from the same period a year earlier .
Мітка:    1 → equivalent


### Знайомимось з BERT BASE UNCASED

Для цього прочитайте інформацію на [картці моделі](https://huggingface.co/google-bert/bert-base-uncased)

In [5]:
from transformers import pipeline

unmasker = pipeline('fill-mask', model='bert-base-uncased')
unmasker("The man worked as a [MASK].")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

[{'score': 0.09747552871704102,
  'token': 10533,
  'token_str': 'carpenter',
  'sequence': 'the man worked as a carpenter.'},
 {'score': 0.052383244037628174,
  'token': 15610,
  'token_str': 'waiter',
  'sequence': 'the man worked as a waiter.'},
 {'score': 0.049627017229795456,
  'token': 13362,
  'token_str': 'barber',
  'sequence': 'the man worked as a barber.'},
 {'score': 0.037886038422584534,
  'token': 15893,
  'token_str': 'mechanic',
  'sequence': 'the man worked as a mechanic.'},
 {'score': 0.037680864334106445,
  'token': 18968,
  'token_str': 'salesman',
  'sequence': 'the man worked as a salesman.'}]

In [6]:
print("Не по темі, але погляньте, яка \"несправедливість\"...")
unmasker("The woman worked as a [MASK].")

Не по темі, але погляньте, яка "несправедливість"...


[{'score': 0.21981368958950043,
  'token': 6821,
  'token_str': 'nurse',
  'sequence': 'the woman worked as a nurse.'},
 {'score': 0.15973998606204987,
  'token': 13877,
  'token_str': 'waitress',
  'sequence': 'the woman worked as a waitress.'},
 {'score': 0.11547233909368515,
  'token': 10850,
  'token_str': 'maid',
  'sequence': 'the woman worked as a maid.'},
 {'score': 0.03796861320734024,
  'token': 19215,
  'token_str': 'prostitute',
  'sequence': 'the woman worked as a prostitute.'},
 {'score': 0.030423680320382118,
  'token': 5660,
  'token_str': 'cook',
  'sequence': 'the woman worked as a cook.'}]

### Токенізація пари речень

BERT приймає обидва речення як одну послідовність: `[CLS] речення 1 [SEP] речення 2 [SEP]`.
Поле `token_type_ids` позначає, до якого речення належить кожен токен (0 або 1).

Примітка: класифікаційний токен (`[CLS]`) використовується як вхідне представлення для задач класифікації в NLP та машинному навчанні. Він акумулює інформацію з усієї вхідної послідовності та передає її крізь шари моделі для подальшої обробки. Потім модель використовує це представлення для формування прогнозів або віднесення вхідних даних до заздалегідь визначених категорій.

In [7]:
checkpoint = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(checkpoint)
encoded = tokenizer(example["sentence1"], example["sentence2"])

print("")
print(tokenizer.convert_ids_to_tokens(encoded["input_ids"])[:20])
print(encoded["token_type_ids"][:20])
print(tokenizer.convert_ids_to_tokens(encoded["input_ids"])[-20:])
print(encoded["token_type_ids"][-20:])


['[CLS]', 'revenue', 'in', 'the', 'first', 'quarter', 'of', 'the', 'year', 'dropped', '15', 'percent', 'from', 'the', 'same', 'period', 'a', 'year', 'earlier', '.']
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[',', 'revenue', 'the', 'first', 'quarter', 'of', 'the', 'year', 'dropped', '15', 'percent', 'from', 'the', 'same', 'period', 'a', 'year', 'earlier', '.', '[SEP]']
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


In [8]:
def tokenize_function(batch):
    # truncation=True обрізає надто довгі пари до максимальної довжини моделі
    return tokenizer(batch["sentence1"], batch["sentence2"], truncation=True)


tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)

# Доповнення (padding) робимо динамічно для кожного батча — до найдовшого прикладу в ньому
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
tokenized_datasets

Map:   0%|          | 0/3668 [00:00<?, ? examples/s]

Map:   0%|          | 0/408 [00:00<?, ? examples/s]

Map:   0%|          | 0/1725 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1725
    })
})

## 2. Модель: кістяк + нова голова

`AutoModelForSequenceClassification` бере кістяк (backbone) BERT і додає нову голову-класифікатор
з `num_labels` виходами. Зверніть увагу на звіт під час завантаження:
- ваги `cls.*` позначені як не використані (`UNEXPECTED`, у старих версіях — «not used»): це відкинута голова попереднього тренування;
- ваги `classifier.*` позначені як відсутні (`MISSING`, у старих версіях — «newly initialized»): це нова голова, ініціалізована випадково.

In [9]:
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

print(model.classifier)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Linear(in_features=768, out_features=2, bias=True)


In [10]:
def count_params(module, trainable_only=False):
    return sum(p.numel() for p in module.parameters() if p.requires_grad or not trainable_only)


print(f"Кістяк (model.bert):   {count_params(model.bert):>12,} параметрів")
print(f"Голова (model.classifier): {count_params(model.classifier):>8,} параметрів")

Кістяк (model.bert):    109,482,240 параметрів
Голова (model.classifier):    1,538 параметрів


## 3. Повне донавчання з `Trainer`

`Trainer` — клас бібліотеки Transformers, який містить увесь цикл тренування й оцінювання.
Метрики рахуємо самі у функції `compute_metrics`.

Оновлюються **всі** ваги: і кістяк, і нова голова.
Для повного донавчання використовують малий learning rate (`2e-5`),
щоб не зруйнувати знання, отримані під час попереднього тренування.

In [11]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(labels, predictions),
    }


training_args = TrainingArguments(
    output_dir="bert-mrpc",
    num_train_epochs=3,             # зазвичай вистачає 1-3 епохи
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    eval_strategy="epoch",          # оцінюємо на валідації після кожної епохи
    save_strategy="no",             # чекпоінти не зберігаємо
    logging_steps=50,
    fp16=torch.cuda.is_available(), # змішана точність пришвидшує тренування на GPU
    report_to="none",
    seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [12]:
print(f"Параметрів, що тренуються: {count_params(model, trainable_only=True):,}")
trainer.train()
trainer.evaluate()

Параметрів, що тренуються: 109,483,778


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.487028,0.381455,0.838235,0.886986
2,0.328676,0.357605,0.848039,0.893471
3,0.167169,0.416541,0.860294,0.901554


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.167169,0.416541,3,0.860294,0.901554


{'eval_loss': 0.4165409803390503,
 'eval_accuracy': 0.8602941176470589,
 'eval_f1': 0.9015544041450777}

## 4. Перевірка на власних прикладах

In [13]:
def predict(model, sentence1, sentence2):
    model.eval()
    inputs = tokenizer(sentence1, sentence2, return_tensors="pt").to(model.device)
    with torch.no_grad():
        probs = model(**inputs).logits.softmax(dim=-1)[0]
    return f"парафраз: {probs[1]:.1%}"


pairs = [
    ("The company reported a profit of $5 million.", "The firm earned five million dollars in profit."),
    ("The company reported a profit of $5 million.", "The weather was sunny all week."),
    ("She bought a new car yesterday.", "Yesterday she purchased a new vehicle."),
    # 4 власні перевірки (п. 10):
    ("The server crashed due to a memory leak.", "A memory leak caused the server to go down."),
    ("The developer fixed the bug before the release.", "The developer failed to fix the bug before the release."),
    ("Ukraine launched a new digital public service.", "Many citizens use smartphones every day."),
    ("He decided to call off the meeting.", "He chose to cancel the meeting."),
]
for s1, s2 in pairs:
    print(f"{s1}\n{s2}\n→ {predict(model, s1, s2)}\n")

The company reported a profit of $5 million.
The firm earned five million dollars in profit.
→ парафраз: 86.0%

The company reported a profit of $5 million.
The weather was sunny all week.
→ парафраз: 2.6%

She bought a new car yesterday.
Yesterday she purchased a new vehicle.
→ парафраз: 99.0%

The server crashed due to a memory leak.
A memory leak caused the server to go down.
→ парафраз: 97.1%

The developer fixed the bug before the release.
The developer failed to fix the bug before the release.
→ парафраз: 99.2%

Ukraine launched a new digital public service.
Many citizens use smartphones every day.
→ парафраз: 31.0%

He decided to call off the meeting.
He chose to cancel the meeting.
→ парафраз: 97.5%



In [14]:
# Експеримент для п. 9: збільшуємо кількість епох з 3 до 8
set_seed(42)
model_8ep = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)
training_args_8ep = TrainingArguments(
    output_dir="bert-mrpc-8ep",
    num_train_epochs=8,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=50,
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=42,
)
trainer_8ep = Trainer(
    model=model_8ep,
    args=training_args_8ep,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)
trainer_8ep.train()
trainer_8ep.evaluate()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.486926,0.377195,0.835784,0.880995
2,0.315664,0.353929,0.850490,0.896435
3,0.155430,0.524629,0.855392,0.890943
4,0.077128,0.643009,0.862745,0.902778
5,0.013990,0.713369,0.860294,0.902564
6,0.012790,0.816343,0.862745,0.904437
7,0.013947,0.805333,0.865196,0.905983
8,0.005038,0.784189,0.862745,0.903114


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.005038,0.784189,8,0.862745,0.903114


{'eval_loss': 0.7841892838478088,
 'eval_accuracy': 0.8627450980392157,
 'eval_f1': 0.903114186851211}

## Відповіді на теоретичні питання (0.1, питання 1–10)

### 1. Як довго виконувалися клітинки ноутбуку `01_bert_head`? Чи були проблеми з запуском, які?
Ноутбук виконувався в Google Colab на GPU **Tesla T4**. Основне тренування на 3 епохах зайняло **1 хв 02 с** (690 кроків), додатковий експеримент на 8 епохах — **2 хв 35 с** (1 840 кроків), а сумарний час виконання всіх клітинок разом із завантаженням датасету MRPC і ваг `bert-base-uncased` склав близько **4–5 хвилин**. Критичних проблем із запуском не було: виводилися лише інформаційне попередження про відсутність `HF_TOKEN` та очікуваний звіт про невикористані ваги `cls.*` (`UNEXPECTED`) і випадково ініціалізовану нову голову `classifier.*` (`MISSING`).

### 2. Як Ви розумієте, що таке токен `[CLS]`? Яким чином він виконує свою функцію?
`[CLS]` (Classification token) — спеціальний службовий токен, який завжди ставиться на початок вхідної послідовності BERT (`[CLS] речення 1 [SEP] речення 2 [SEP]`). Оскільки в архітектурі BERT використовується двонаправлена увага (bidirectional self-attention), на кожному шарі трансформера вектор токена `[CLS]` взаємодіє з усіма іншими токенами обох речень і поступово акумулює узагальнене семантичне представлення всієї пари речень. На виході кістяка фінальний вектор `[CLS]` (розмірності 768) спочатку проходить через шар `bert.pooler` (`Linear(768, 768)` з активацією `tanh`) та `Dropout`, після чого передається в голову-класифікатор (`Linear(768, 2)`) для обчислення логітів класів.

### 3. Яке пряме застосування базової моделі `bert-base-uncased`?
`bert-base-uncased` — це попередньо натренована енкодерна модель (base-модель), навчена на двох задачах: Masked Language Modeling (MLM — передбачення замаскованих токенів `[MASK]`) та Next Sentence Prediction (NSP — визначення, чи є друге речення логічним продовженням першого). Її пряме застосування без донавчання — заповнення пропусків у тексті (`fill-mask`) та отримання контекстних векторних представлень (ембедингів) слів і речень; проте на практиці її головне призначення — слугувати кістяком (backbone) для класичного донавчання під прикладні NLP-задачі (класифікація тексту, розпізнавання іменованих сутностей NER, пошук відповіді в контексті QA).

### 4. Як Ви вважаєте, чи варто боротися (і якщо так, то яким чином) з проявами сексизму LLM-моделей?
Так, із проявами гендерних упереджень варто боротися, особливо коли моделі використовуються в прикладних системах, що впливають на рішення щодо людей (рекрутинг, скоринг, модерація, підтримка користувачів), оскільки моделі відтворюють і посилюють історичні стереотипи з навчальних корпусів (як видно на прикладі `fill-mask` для *man* vs *woman*). Боротися з цим доцільно комплексно:
- очищенням і балансуванням навчальних датасетів (data curation / counterfactual data augmentation);
- етапом вирівнювання (RLHF або DPO) на парах відповідей, що заохочує нейтральні та неупереджені формулювання;
- регулярним тестуванням на бенчмарках упередженості та red-teaming перед релізом;
- системними інструкціями та пост-перевіркою виводу в продакшн-застосунках, уникаючи при цьому надмірної цензури, яка спотворює історичні або фактичні запити.

### 5. Для чого в секції «Токенізація пари речень» виводяться числа 0 і 1?
Це значення масиву `token_type_ids` (сегментні індекси). Оскільки BERT отримує два речення як одну об'єднану послідовність, `0` позначає всі токени, що належать першому реченню (включно з `[CLS]` та першим `[SEP]`), а `1` — токени другого речення (включно з другим `[SEP]`). Це допомагає моделі чітко розрізняти межі між першим і другим реченням під час порівняння їхнього змісту.

### 6. Що позначають числа 768 і 2, що виводяться після завантаження `AutoModelForSequenceClassification.from_pretrained()`?
Рядок `Linear(in_features=768, out_features=2, bias=True)` описує новий лінійний шар голови-класифікатора:
- **768** (`in_features`) — розмірність прихованого вектора (hidden size) моделі `bert-base-uncased`, тобто довжина вектора токена `[CLS]`, який надходить на вхід класифікатора;
- **2** (`out_features`) — кількість вихідних логітів, що дорівнює `num_labels=2` (два класи: `0` — не парафраз, `1` — парафраз).

### 7. Те, що в кістяку (backbone) набагато більше параметрів, ніж у голові, це добре чи погано в нашому випадку?
У нашому випадку це **добре**. Кістяк (`~109,5 млн` параметрів) уже навчився розуміти граматику, лексику й семантику англійської мови під час масштабного попереднього тренування (transfer learning). Невелика голова (`1 538` параметрів: `768 × 2 + 2`) лише відображає готове 768-вимірне представлення на 2 класи. Завдяки цьому модель швидко сходиться й показує високу якість на порівняно малому датасеті MRPC (~3 668 тренувальних пар) без необхідності збирати мільйони розмічених прикладів.

### 8. Що саме вимірюється за допомогою метрик accuracy & F1 в `01_bert_head`?
- **Accuracy** — частка всіх пар речень на валідаційній вибірці, для яких модель правильно визначила клас (і парафрази, і не-парафрази).
- **F1-score** — гармонійне середнє між точністю (*precision*) і повнотою (*recall*) для позитивного класу (`1` — парафраз). Використання F1 важливе, оскільки датасет MRPC незбалансований (~67% пар є парафразами), тому сама лише accuracy може створювати хибне враження високої якості навіть для моделі, що схильна занадто часто передбачати домінуючий клас.

### 9. Збільшіть кількість епох тренування в `01_bert_head`, покажіть (скопіювавши результати) що відбувається і чому.
*Скопійовані результати тренування на 8 епохах (`trainer_8ep.train()`, 1 840 кроків, 02:35):*

| Epoch | Training Loss | Validation Loss | Accuracy | F1 |
| --- | --- | --- | --- | --- |
| 1 | 0.486926 | 0.377195 | 0.835784 | 0.880995 |
| 2 | 0.315664 | 0.353929 | 0.850490 | 0.896435 |
| 3 | 0.155430 | 0.524629 | 0.855392 | 0.890943 |
| 4 | 0.077128 | 0.643009 | 0.862745 | 0.902778 |
| 5 | 0.013990 | 0.713369 | 0.860294 | 0.902564 |
| 6 | 0.012790 | 0.816343 | 0.862745 | 0.904437 |
| 7 | 0.013947 | 0.805333 | 0.865196 | 0.905983 |
| 8 | 0.005038 | 0.784189 | 0.862745 | 0.903114 |

**Пояснення:** Оптимальний `Validation Loss` досягається вже на **2-й епосі (`0.3539`)**. Далі `Training Loss` стрімко падає майже до нуля (`0.0050` на 8-й епосі), тоді як `Validation Loss` зростає більш ніж удвічі (`0.8163` на 6-й епосі, `0.7842` на 8-й), а `Accuracy` і `F1` фактично виходять на плато (~`0.86` і ~`0.90`). Це класичне **перенавчання (overfitting)**: модель на ~109,5 млн параметрів запам'ятовує 3 668 тренувальних пар MRPC замість кращого узагальнення, а її прогнози на помилкових валідаційних прикладах стають надмірно самовпевненими, через що різко зростає крос-ентропійна функція втрат.

### 10. Додайте ще 4 перевірки на власних даних в пункті 4 в `01_bert_head`.
4 власні перевірки додано в кодову клітинку секції 4. Результати моделі (3 епохи):
1. *"The server crashed due to a memory leak."* / *"A memory leak caused the server to go down."* → **парафраз: 97.1%** (правильно: розпізнано синонімічну перестановку причини й наслідку).
2. *"The developer fixed the bug before the release."* / *"The developer failed to fix the bug before the release."* → **парафраз: 99.2%** (показова **помилка моделі**: через майже повний лексичний збіг слів BERT проігнорував заперечення *failed to*, що демонструє типові обмеження класифікаторів, донавчених на малому новинному корпусі MRPC).
3. *"Ukraine launched a new digital public service."* / *"Many citizens use smartphones every day."* → **парафраз: 31.0%** (правильно: не парафраз).
4. *"He decided to call off the meeting."* / *"He chose to cancel the meeting."* → **парафраз: 97.5%** (правильно: модель коректно зіставила фразове дієслово *call off* із *cancel*).